# Creating configurations — every variant

Configurations let one part file hold several design variants, each
able to suppress features or override parameters independently of
the others.

**Prereq:** open a fresh empty part in Alibre.

## Setup: a part with two features (Block + Fillet)

In [ ]:
from alibrex import (
    CurrentPart,
    connect,
    ADDirectionType,
    ADPartFeatureEndCondition,
)

part = CurrentPart()
xy = part.DesignPlanes.Item(0)
sk = part.Sketches.AddSketch(None, xy, "Base")
figs = sk.Figures
figs.AddLine(0.0, 0.0, 4.0, 0.0)
figs.AddLine(4.0, 0.0, 4.0, 3.0)
figs.AddLine(4.0, 3.0, 0.0, 3.0)
figs.AddLine(0.0, 3.0, 0.0, 0.0)
part.Features.AddExtrudedBoss(
    sk, 2.0, ADPartFeatureEndCondition.AD_TO_DEPTH,
    None, None, 0.0,
    ADDirectionType.AD_ALONG_NORMAL, None, None, False,
    None, False,
    "Block", "Depth", "",
)

## Add a fillet on every edge

In [ ]:
root = connect()
body = part.Bodies.Item(0)
edges = root.NewObjectCollector()
for i in range(body.Edges.Count):
    edges.Add(body.Edges.Item(i))
fillet = part.Features.AddConstantRadiusFilletFeature(
    edges, 0.2, True, "", "Fillet",
)
fillet.Name

## Initial configuration state

In [ ]:
configs = part.Configurations
print(f"Count = {configs.Count}")
print(f"Active = {part.ActiveConfiguration.Name}")

## `AddConfiguration(name, locked)` — create an unlocked variant

`locked=False` means the configuration is editable — you can change
feature suppression or parameter overrides on it.

In [ ]:
cfg_small = configs.AddConfiguration("Small", False)
cfg_small.Name, cfg_small.ID

## Locked configuration

`locked=True` freezes the configuration — useful for shipping a
design variant that should not be edited downstream.

In [ ]:
cfg_shipped = configs.AddConfiguration("Shipped", True)
cfg_shipped.Name, cfg_shipped.ID

## Switch the active configuration

In [ ]:
part.ActiveConfiguration = cfg_small
part.ActiveConfiguration.Name

In [ ]:
part.ActiveConfiguration = cfg_shipped
part.ActiveConfiguration.Name

## Suppress a feature in one config only

In [ ]:
part.ActiveConfiguration = cfg_small
fillet.Suppressed = True
part.RegenerateAll()
fillet.Suppressed

Switch back — fillet should still be unsuppressed there.

In [ ]:
part.ActiveConfiguration = cfg_shipped
part.RegenerateAll()
fillet.Suppressed

## Iterate every configuration

In [ ]:
for i in range(configs.Count):
    c = configs.Item(i)
    print(f"  {c.Name:20s}  id={c.ID}")

## Summary

| Action | Method |
|---|---|
| Add editable config | `configs.AddConfiguration(name, False)` |
| Add locked config | `configs.AddConfiguration(name, True)` |
| Switch active | `part.ActiveConfiguration = cfg` |
| Per-config feature suppress | switch active, set `feature.Suppressed = True` |
| Count | `configs.Count` |
| Iterate | `configs.Item(i)` for `i in range(configs.Count)` |
| Read current | `part.ActiveConfiguration.Name` |